# JP-1 C0：凍結前の確かめ（CPU だけ・ベンチマークは読みません）

計画書 `notes/jp1_sarashina_gate.md`（第3版）のゲート C0 です。

- Sarashina2.2（1B と 3B）と MGSM の、Hugging Face の今の版（コミット）とライセンスの表示、使う道具の版を調べて記録します。**MGSM は中身を読まず、ファイルの一覧だけを見ます。**
- 計画書の付録C（語の種類の例文）と付録D（答えの読み取りの例）を、本物の tokenizer・fugashi・NLTK で動かして、付録に書いた答えと比べます。例文は計画書のために作った無害な文です。

**使い方**：ランタイムは **CPU**。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で 5〜10 分くらい。結果は Drive の `DECISION_POINTS/jp1/C0/JP1_C0_result_<コミットの頭7文字>.json`。終わったら Claude に教えてね（Claude が読んで、食い違いがあれば直します）。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 C0（凍結前の確かめ・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 C0', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_c0'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "5dfbf0eb26c9936f3cef8e32751cf9ddc84447b2"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
D = Path("/content/drive/MyDrive/DECISION_POINTS/jp1/C0")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
if shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0:
    print("⚠️ GPU ランタイムになっています。C0 は CPU だけで足ります（CU の節約のため、CPU に切り替えてね）。")

In [ ]:
# Cell 2 — packages and the versions to lock (nothing from MGSM is downloaded; metadata only).
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fugashi", "unidic-lite", "nltk", "sentencepiece", "protobuf"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
PKGS = ["transformers", "tokenizers", "torch", "huggingface_hub", "fugashi", "unidic-lite", "nltk", "sentencepiece"]
VERSIONS = {p: md.version(p) for p in PKGS}
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as _sw
SW_PATH = Path(nltk.data.find("corpora/stopwords/english"))
STOPWORDS = frozenset(w.strip().lower() for w in SW_PATH.read_text(encoding="utf-8").splitlines() if w.strip())
STOPWORDS_FILE = {"sha256": hashlib.sha256(SW_PATH.read_bytes()).hexdigest(), "words": len(STOPWORDS)}
assert STOPWORDS == frozenset(_sw.words("english")), "NLTK のファイルと words() が食い違っています"

from huggingface_hub import HfApi
api = HfApi()
MODELS = {"1b": "sbintuitions/sarashina2.2-1b-instruct-v0.1", "3b": "sbintuitions/sarashina2.2-3b-instruct-v0.1"}
DATASET = "juletxara/mgsm"
def card_license(info):
    cd = getattr(info, "card_data", None) or getattr(info, "cardData", None)
    if cd is None:
        return None
    d = cd.to_dict() if hasattr(cd, "to_dict") else dict(cd)
    return d.get("license")
HF = {}
for k, rid in MODELS.items():
    info = api.model_info(rid, token=False)
    HF[k] = {"repo": rid, "sha": info.sha, "card_license": card_license(info),
             "files": sorted(s.rfilename for s in (info.siblings or []))}
dinfo = api.dataset_info(DATASET, token=False)
dfiles = sorted(s.rfilename for s in (dinfo.siblings or []))
HF["mgsm"] = {"repo": DATASET, "sha": dinfo.sha, "card_license": card_license(dinfo),
              "files_ja_en": [f for f in dfiles if f.split("/")[0] in ("ja", "en")],
              "files_total": len(dfiles)}

from transformers import AutoTokenizer
TOK, TOKINFO = {}, {}
for k, rid in MODELS.items():
    tok = AutoTokenizer.from_pretrained(rid, revision=HF[k]["sha"], token=False)
    TOK[k] = tok
    vocab = sorted(tok.get_vocab().items())
    TOKINFO[k] = {"class": type(tok).__name__, "vocab_size": len(vocab),
                  "vocab_sha256": hashlib.sha256(json.dumps(vocab, ensure_ascii=False).encode("utf-8")).hexdigest(),
                  "chat_template_sha256": hashlib.sha256((tok.chat_template or "").encode("utf-8")).hexdigest(),
                  "has_chat_template": bool(tok.chat_template)}
SAME_TOKENIZER = TOKINFO["1b"]["vocab_sha256"] == TOKINFO["3b"]["vocab_sha256"]
print(json.dumps({"versions": VERSIONS, "stopwords": STOPWORDS_FILE,
                  "hf": {k: {"sha": v["sha"], "card_license": v["card_license"]} for k, v in HF.items()},
                  "mgsm_ja_en_files": HF["mgsm"]["files_ja_en"], "same_tokenizer_1b_3b": SAME_TOKENIZER}, indent=1, ensure_ascii=False))

In [ ]:
# Cell 3 — unit tests, appendix D (readout), appendix C (categories with the real tokenizer, fugashi and NLTK).
r = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_jp1_text.py"],
                   capture_output=True, text=True)
UNIT = {"returncode": r.returncode, "tail": r.stderr.strip().splitlines()[-3:]}
print("unit tests:", "OK" if r.returncode == 0 else "FAILED", UNIT["tail"])
from pork import jp1_text as J
from pork import dp1_forks as dp
import fugashi
TAGGER = fugashi.Tagger()
UNIDIC = {"dicdir_name": Path(__import__("unidic_lite").DICDIR).name}

D_ROWS = J.check_appendix_d()
C_RESULTS, A_TOKEN = {}, {}
for k in (["1b"] if SAME_TOKENIZER else ["1b", "3b"]):
    tok = TOK[k]
    dec = lambda ids, tok=tok: tok.decode(list(ids), skip_special_tokens=True)
    rows = []
    for cid, lang, sent, specs in J.APPENDIX_C:
        ids = tok(sent, add_special_tokens=False)["input_ids"]
        T, words, toks = J.classify(lang, ids, dec, stopwords=STOPWORDS, tagger=TAGGER)
        rows.append(J.check_sentence(cid, lang, sent, specs, T, words, toks))
    C_RESULTS[k] = {"rows": rows, "verdict": J.appendix_c_verdict(rows)}
    agree, n = 0, 0
    for i, lang, text, val, reason, start in J.APPENDIX_D:
        if start is None:
            continue
        ids = tok(text, add_special_tokens=False)["input_ids"]
        T, spans = J.token_spans(ids, dec)
        if T != text:
            continue
        n += 1
        agree += int(dp.answer_first_token(ids, dec, start) == J.answer_token_ref(spans, start))
    A_TOKEN[k] = {"checked": n, "agree": agree}

JA_MORPHEMES = {cid: [[w.start, w.surface, w.lemma, w.category] for w in J.ja_words(sent, TAGGER)]
                for cid, lang, sent, specs in J.APPENDIX_C if lang == "ja"}   # the same per-line path as the analysis

# chat template sanity (harmless text): the prompt goes in as the only user turn
CHAT_OK = {}
for k in TOK:
    for lang, tmpl in (("ja", J.PROMPT_JA), ("en", J.PROMPT_EN)):
        txt = TOK[k].apply_chat_template([{"role": "user", "content": tmpl.format(question="1+1")}], tokenize=False,
                                         add_generation_prompt=True)
        CHAT_OK[f"{k}_{lang}"] = bool(txt) and tmpl.format(question="1+1") in txt

D_PASS = all(x["ok"] for x in D_ROWS)
C_PASS = all(v["verdict"]["pass"] for v in C_RESULTS.values())
A_PASS = all(v["checked"] > 0 and v["agree"] == v["checked"] for v in A_TOKEN.values())
print("appendix D:", sum(x["ok"] for x in D_ROWS), "/", len(D_ROWS), "  answer token a:", A_TOKEN)
for k, v in C_RESULTS.items():
    print(f"appendix C [{k}]:", v["verdict"])
    for row in v["rows"]:
        if not row["ok"]:
            print("  ✗", row["id"], {x: row[x] for x in ("text_roundtrip", "word_mismatch", "token_mismatch", "unclassified")})
print("chat template:", CHAT_OK)

In [ ]:
# Cell 4 — save the result (small JSON on Drive), then notify and disconnect.
ALL_PASS = bool(UNIT["returncode"] == 0 and D_PASS and C_PASS and A_PASS and all(CHAT_OK.values()))
RESULT = {"schema": "jp1_c0_v1", "head": EXPECTED_HEAD, "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds"),
          "versions": VERSIONS, "stopwords_file": STOPWORDS_FILE, "unidic": UNIDIC, "hf": HF, "tokenizers": TOKINFO,
          "same_tokenizer_1b_3b": SAME_TOKENIZER, "unit_tests": UNIT, "appendix_d": D_ROWS, "appendix_c": C_RESULTS,
          "answer_token": A_TOKEN, "chat_template": CHAT_OK, "ja_morphemes": JA_MORPHEMES,
          "pass": {"unit": UNIT["returncode"] == 0, "appendix_d": D_PASS, "appendix_c": C_PASS, "answer_token": A_PASS,
                   "chat_template": all(CHAT_OK.values()), "all": ALL_PASS}}
path = D / f"JP1_C0_result_{EXPECTED_HEAD[:7]}.json"   # 回ごとに別の名前（前の回を上書きしない）
path.write_text(json.dumps(RESULT, ensure_ascii=False, indent=1, sort_keys=True), encoding="utf-8")
print("保存しました：", path, f"({path.stat().st_size} bytes)")
print("C0 の結果：", "すべて付録のとおり ✅" if ALL_PASS else "食い違いがあります（Claude が見て直します）")
notify("C0 終わりました：" + ("すべて付録のとおり" if ALL_PASS else "食い違いあり（Claude に見せてね）"), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()